# 06 — Volatility Surface (Layers 0–5)

Forward/discount recovery + our own **Black-76 implied vols** + an **arbitrage-free SVI smile** + **trader-standard vol signals** + **variance risk premium** (implied vs realized).

**Run after Notebook 02** on the same day. It reads the **same raw chain CSVs** that 02 already downloaded — no extra scraping, and participant OI is not needed here. It processes today's chains for the active symbols, prints a forward/IV health check vs NSE, fits a clean SVI smile per expiry, and saves everything so history accumulates.

- **Layer 0** — mid prices + liquidity filters, OTM wing only
- **Layer 1** — forward `F` and discount factor `DF` from put-call parity `C − P = DF·(F − K)`
- **Layer 2** — our implied vol via Black-76 (self-consistent smile; matches Kite/Groww/Sensibull)
- **Layer 3** — raw **SVI** smile `w(k) = a + b(ρ(k−m) + √((k−m)² + σ²))` fit per expiry, with static no-arbitrage checks (min-variance, Gatheral butterfly `g(k) ≥ 0`, and calendar across expiries)
- **Layer 4** — distil each smile into trader-standard signals: **ATM IV** (level), **skew** (ATM slope + 25Δ risk reversal), **convexity** (25Δ butterfly), and a **constant-maturity term structure** (7d/30d ATM IV + slope) — the fixed horizons Layer 5 will score against realised vol for VRP
- **Layer 5** — **variance risk premium**: our implied IV vs **realized vol** (from index OHLC), both per-expiry (implied at t vs realised over t→expiry) and fixed-horizon (cmt7/30 vs forward realised). VRP = implied − realized; positive = premium sellers' edge, negative = options were cheap

NSE's published IV is computed off spot per-leg, so its call/put IVs disagree at the same strike. Ours is forward-consistent, so the `mean_abs_diff_vs_nse` below is expected to be ~1–2 vol points, not zero — that gap is NSE's artifact, not ours.

The SVI fit uses liquidity weights on an adaptive ±4σ moneyness window (short expiries would otherwise be swamped by junk far-OTM strikes). Typical fit RMSE is ~0.1–0.5 vol points. Butterfly/min-variance are judged over the traded region; a fit that fails is flagged via `no_arb_ok` so we don't trust it.

**Layer 5 needs the underlying OHLC** — run `python scripts/fetch_underlying.py` before this notebook so realized vol is up to date. Realized uses close-to-close (includes overnight gaps, matching what option premium prices); Parkinson/GK/RS are intraday-only so they're not used for VRP.

In [21]:
import os
import sys
import glob
import json
from datetime import datetime

import pandas as pd
import yaml

PROJECT_ROOT = os.path.abspath(os.path.join(os.getcwd(), ".."))
sys.path.insert(0, PROJECT_ROOT)

from src.vol_model import analyze_day
from src.trading_calendar import resolve_session_date, describe_session

with open(os.path.join(PROJECT_ROOT, "config", "config.yaml")) as f:
    CONFIG = yaml.safe_load(f)

SESSION_OVERRIDE = None  # e.g. "2026-10-01" to reprocess a past session
TODAY = resolve_session_date(SESSION_OVERRIDE)
print(describe_session())
DATA_RAW = os.path.join(PROJECT_ROOT, "data", "raw")
VOL_DIR = os.path.join(PROJECT_ROOT, "data", "processed", "vol_surface")
os.makedirs(VOL_DIR, exist_ok=True)

ACTIVE_SYMBOLS = CONFIG.get("collection", {}).get("active_symbols", ["NIFTY", "BANKNIFTY"])

pd.set_option("display.width", 160)
pd.set_option("display.max_rows", 200)
pd.set_option("display.float_format", lambda v: f"{v:0.4f}")

print(f"Vol date: {TODAY} | Symbols: {ACTIVE_SYMBOLS}")

Session: 2026-10-07 (Wed) | now 2026-10-08 00:57 IST | 2026-10-08 is not a finished trading session
Vol date: 2026-10-07 | Symbols: ['NIFTY', 'BANKNIFTY']


In [22]:
def chain_files_for_date(symbol, run_date):
    pattern = os.path.join(DATA_RAW, symbol.lower(), "option_chain", f"{run_date}_*.csv")
    return [p for p in sorted(glob.glob(pattern)) if "Select" not in os.path.basename(p)]


def spread_cap_for(dte):
    """Thin / back-series expiries get a tighter liquidity gate."""
    return 0.20 if (dte or 0) > 30 else 0.35


def process_symbol_vol(symbol, run_date):
    health_rows = []
    for csv in chain_files_for_date(symbol, run_date):
        meta_path = csv.replace(".csv", ".meta.json")
        dte = None
        if os.path.exists(meta_path):
            with open(meta_path) as fh:
                dte = json.load(fh).get("dte")

        res = analyze_day(csv, max_spread_pct=spread_cap_for(dte))
        if res.get("skipped"):
            print(f"  [skip] {res.get('expiry')} dte={res.get('dte')}: {res['skipped']}")
            continue

        s = res["summary"]
        health_rows.append({
            "symbol": res["symbol"], "date": res["date"], "expiry": res["expiry"],
            "dte": res["dte"], "spot": res["spot"], "forward": res["forward"],
            "discount_factor": res["discount_factor"], "parity_r2": res["parity_r2"],
            "parity_pairs": res["parity_pairs"], "n_liquid": s["n_liquid"],
            "mean_abs_diff_vs_nse": s["mean_abs_diff_liquid"], "corr_vs_nse": s["corr_liquid"],
        })

        points = res["reconciliation"].copy()
        points.insert(0, "expiry", res["expiry"])
        points.insert(0, "date", res["date"])
        points.insert(0, "symbol", res["symbol"])
        dst = os.path.join(VOL_DIR, symbol.lower())
        os.makedirs(dst, exist_ok=True)
        points.to_parquet(os.path.join(dst, f"{res['date']}_{res['expiry']}.parquet"), index=False)

    return pd.DataFrame(health_rows)

In [23]:
# ── RUN (session only, active symbols) ─────────────
_missing = [s for s in ACTIVE_SYMBOLS if not chain_files_for_date(s, TODAY)]
if _missing:
    raise FileNotFoundError(
        f"No raw chains for session {TODAY} for {_missing}. Run notebook 02 first "
        "(or set SESSION_OVERRIDE to reprocess an older session)."
    )
for sym in ACTIVE_SYMBOLS:
    _px = os.path.join(DATA_RAW, "underlying", f"{sym.lower()}.csv")
    _last = pd.read_csv(_px, usecols=["date"])["date"].max() if os.path.exists(_px) else None
    if _last != TODAY:
        print(f"[WARN] {sym} prices end at {_last}, not {TODAY}: run scripts/fetch_underlying.py "
              "first or realised vol / VRP will be a day stale.")

all_health = []
for sym in ACTIVE_SYMBOLS:
    print("=" * 80)
    print(sym)
    h = process_symbol_vol(sym, TODAY)
    if h.empty:
        print("  no usable expiries")
        continue
    cols = ["expiry", "dte", "spot", "forward", "discount_factor",
            "parity_r2", "n_liquid", "mean_abs_diff_vs_nse", "corr_vs_nse"]
    print(h[cols].to_string(index=False))
    all_health.append(h)

if all_health:
    health = pd.concat(all_health, ignore_index=True)
    health_path = os.path.join(VOL_DIR, f"_health_{TODAY}.csv")
    health.to_csv(health_path, index=False)
    print("\nSaved vol health ->", health_path)
else:
    print("\nNo vol data written for", TODAY)

NIFTY
     expiry  dte       spot    forward  discount_factor  parity_r2  n_liquid  mean_abs_diff_vs_nse  corr_vs_nse
03-Nov-2026   27 22603.0500 22653.6418           0.9869     1.0000        60                1.3713       0.8885
09-Nov-2026   33 22603.0500 22686.8725           0.9777     0.9996        13                1.3976       0.8117
13-Oct-2026    6 22603.0500 22570.2421           0.9980     1.0000       107                1.1037       0.9853
19-Oct-2026   12 22603.0500 22595.7145           0.9946     1.0000        87                1.1191       0.9715
23-Nov-2026   47 22603.0500 22728.2491           0.9833     1.0000        83                1.4655       0.8805
27-Oct-2026   20 22603.0500 22622.1586           0.9900     1.0000        92                1.1809       0.9525
BANKNIFTY
     expiry  dte       spot    forward  discount_factor  parity_r2  n_liquid  mean_abs_diff_vs_nse  corr_vs_nse
23-Nov-2026   47 55055.5500 55574.0044           0.9851     0.9998        81            

In [24]:
# ── Inspect: our clean IV smile for the front expiry of the first symbol ──
sample = sorted(glob.glob(os.path.join(VOL_DIR, ACTIVE_SYMBOLS[0].lower(), f"{TODAY}_*.parquet")))
if sample:
    df = pd.read_parquet(sample[0])
    liq = df[df["liquid"]].sort_values("moneyness")
    print(os.path.basename(sample[0]))
    print(liq[["strike", "moneyness", "otm_side", "our_iv", "nse_iv", "iv_diff", "oi"]].to_string(index=False))
else:
    print("No vol_surface parquet for", TODAY)

2026-10-07_03-Nov-2026.parquet
    strike  moneyness otm_side  our_iv  nse_iv  iv_diff        oi
20500.0000    -0.0999       PE 20.4063 21.2700  -0.8637  509.0000
20800.0000    -0.0854       PE 18.6842 19.4700  -0.7858   36.0000
20950.0000    -0.0782       PE 18.0751 20.0400  -1.9649   19.0000
21000.0000    -0.0758       PE 18.0624 18.8900  -0.8276  497.0000
21300.0000    -0.0616       PE 16.5576 17.8400  -1.2824   52.0000
21400.0000    -0.0569       PE 16.2028 17.5100  -1.3072  233.0000
21500.0000    -0.0523       PE 16.2266 17.2800  -1.0534  535.0000
21550.0000    -0.0499       PE 15.7175 16.9500  -1.2325   83.0000
21600.0000    -0.0476       PE 15.7931 16.8400  -1.0469  141.0000
21650.0000    -0.0453       PE 15.7097 16.5600  -0.8503   45.0000
21700.0000    -0.0430       PE 15.5066 16.5900  -1.0834  316.0000
21750.0000    -0.0407       PE 15.1386 16.3900  -1.2514   32.0000
21800.0000    -0.0384       PE 15.0415 16.3500  -1.3085  210.0000
21850.0000    -0.0361       PE 15.0672 16.810

## Layer 3 — SVI smile fit + no-arbitrage checks

Fits raw SVI to the clean Layer-2 IVs (one smile per expiry), then verifies static no-arbitrage. Saves params + diagnostics to `vol_surface/svi/{symbol}_{date}.csv` and rebuilds the master `vol_surface/svi/_svi_all.csv`.

Read the health table like this: `rmse_vol` is the fit error in vol points (want ≲0.5); `rho` is the smile skew (negative = downside skew, normal for indices); `min_g_data` is Gatheral's butterfly margin over the traded strikes (want ≥0); `no_arb_ok` combines butterfly + min-variance; `calendar_ok` checks total variance is non-decreasing across expiries over their common strike range.

In [25]:
# ── Layer 3: fit SVI per expiry, check no-arb, persist ──────────
from src.svi import fit_smile_from_points, calendar_check
from src.expiry_utils import compute_dte

SVI_DIR = os.path.join(VOL_DIR, "svi")
os.makedirs(SVI_DIR, exist_ok=True)


def fit_symbol_smiles(symbol, run_date):
    rows, fits_for_cal = [], []
    pqs = sorted(glob.glob(os.path.join(VOL_DIR, symbol.lower(), f"{run_date}_*.parquet")))
    for pq in pqs:
        pts = pd.read_parquet(pq)
        expiry = pts["expiry"].iloc[0]
        dte = compute_dte(run_date, expiry)
        res = fit_smile_from_points(pts, dte)
        if not res.get("ok"):
            print(f"  [skip] {expiry} dte={dte}: {res.get('reason')}")
            continue
        p = res["params"]
        rows.append({
            "symbol": symbol.upper(), "date": run_date, "expiry": expiry,
            "dte": dte, "T": res["T"], "n_points": res["n_points"],
            "fit_band": res.get("fit_band"), "rmse_vol": res["rmse_vol"],
            "max_abs_vol": res["max_abs_vol"], "atm_iv": res["atm_iv"] * 100.0,
            "a": p["a"], "b": p["b"], "rho": p["rho"], "m": p["m"], "sigma": p["sigma"],
            "min_variance": res["min_variance"], "min_g_data": res["min_g_data"],
            "min_g_wide": res["min_g_wide"], "butterfly_ok": res["butterfly_ok"],
            "min_variance_ok": res["min_variance_ok"], "no_arb_ok": res["no_arb_ok"],
        })
        fits_for_cal.append((res["T"], p, res.get("fit_band", 0.10)))
    cal = calendar_check(fits_for_cal)
    for r in rows:
        r["calendar_ok"] = cal["calendar_ok"]
        r["calendar_min_gap"] = cal["min_gap"]
    return pd.DataFrame(rows), cal


for sym in ACTIVE_SYMBOLS:
    print("=" * 80)
    print(sym)
    day, cal = fit_symbol_smiles(sym, TODAY)
    if day.empty:
        print("  no smiles fitted")
        continue
    day.to_csv(os.path.join(SVI_DIR, f"{sym.lower()}_{TODAY}.csv"), index=False)
    cols = ["expiry", "dte", "n_points", "fit_band", "rmse_vol", "max_abs_vol",
            "atm_iv", "rho", "min_g_data", "no_arb_ok"]
    print(day[cols].to_string(index=False))
    print(f"  calendar_ok={cal['calendar_ok']} min_gap={cal['min_gap']:.6f}")

# rebuild master from all per-day CSVs so history stays consistent
per_day = [p for p in sorted(glob.glob(os.path.join(SVI_DIR, "*_*.csv")))
           if not os.path.basename(p).startswith("_")]
if per_day:
    master = pd.concat([pd.read_csv(p) for p in per_day], ignore_index=True)
    master = master.sort_values(["symbol", "date", "dte"])
    master.to_csv(os.path.join(SVI_DIR, "_svi_all.csv"), index=False)
    print(f"\nSaved SVI params -> {os.path.join(SVI_DIR, '_svi_all.csv')} ({len(master)} smiles)")

NIFTY
     expiry  dte  n_points  fit_band  rmse_vol  max_abs_vol  atm_iv     rho  min_g_data  no_arb_ok
03-Nov-2026   27        63    0.1065    0.1592       0.8203 13.1131  0.3863      0.0146       True
09-Nov-2026   33        25    0.0445    0.0843       0.2408 13.4064 -0.9999      0.4568       True
13-Oct-2026    6        56    0.0614    0.3003       1.0798 11.8497  0.3047      0.0302       True
19-Oct-2026   12        77    0.0900    0.2743       0.8094 12.4032  0.1738      0.0418       True
23-Nov-2026   47        83    0.1383    0.1168       0.3710 13.1913  0.0931      0.1303       True
27-Oct-2026   20        97    0.1207    0.2394       0.6290 12.8942  0.2093      0.0272       True
  calendar_ok=True min_gap=0.000134
BANKNIFTY
     expiry  dte  n_points  fit_band  rmse_vol  max_abs_vol  atm_iv    rho  min_g_data  no_arb_ok
23-Nov-2026   47        81    0.1676    0.1245       0.5678 13.7944 0.3200      0.0021       True
27-Oct-2026   20       101    0.1314    0.2395       1.2365

In [26]:
# ── Inspect: observed IV vs SVI fit for the front expiry of the first symbol ──
import numpy as np
from src.svi import raw_svi

sym0 = ACTIVE_SYMBOLS[0].lower()
svi_today = os.path.join(SVI_DIR, f"{sym0}_{TODAY}.csv")
sample = sorted(glob.glob(os.path.join(VOL_DIR, sym0, f"{TODAY}_*.parquet")))
if os.path.exists(svi_today) and sample:
    sp = pd.read_csv(svi_today)
    df = pd.read_parquet(sample[0])
    expiry = df["expiry"].iloc[0]
    row = sp[sp["expiry"] == expiry].iloc[0]
    p = {k: float(row[k]) for k in ("a", "b", "rho", "m", "sigma")}
    T = float(row["T"])

    band = float(row["fit_band"])
    liq = df[df["liquid"] & df["our_iv"].notna()].sort_values("moneyness").copy()
    liq["svi_iv"] = np.sqrt(np.maximum(raw_svi(liq["moneyness"].to_numpy(), **p), 1e-12) / T) * 100.0
    liq["resid"] = liq["our_iv"] - liq["svi_iv"]

    print(f"{ACTIVE_SYMBOLS[0]} {expiry}  dte={int(row['dte'])}  "
          f"ATM(SVI,k=0)={row['atm_iv']:.2f}  rho={row['rho']:.3f}  "
          f"RMSE={row['rmse_vol']:.3f}vp  no_arb_ok={bool(row['no_arb_ok'])}")
    # show the region the fit actually used (|k| <= fit_band); far wings are
    # excluded from the fit on purpose (junk for a short expiry) so their large
    # residuals here are extrapolation, not fit error
    inband = liq[liq["moneyness"].abs() <= band].copy()
    step = max(1, len(inband) // 25)
    print(f"in-fit region |k|<={band:.3f}  ({len(inband)} strikes; "
          f"{len(liq) - len(inband)} liquid wing strikes excluded from fit)")
    print(inband.iloc[::step][["strike", "moneyness", "otm_side", "our_iv", "svi_iv", "resid"]]
          .to_string(index=False))
else:
    print("No SVI fit to inspect for", TODAY)

NIFTY 03-Nov-2026  dte=27  ATM(SVI,k=0)=13.11  rho=0.386  RMSE=0.159vp  no_arb_ok=True
in-fit region |k|<=0.107  (62 strikes; 1 liquid wing strikes excluded from fit)
    strike  moneyness otm_side  our_iv  svi_iv   resid
20500.0000    -0.0999       PE 20.4063 19.5859  0.8203
20950.0000    -0.0782       PE 18.0751 18.0432  0.0320
21300.0000    -0.0616       PE 16.5576 16.8675 -0.3099
21500.0000    -0.0523       PE 16.2266 16.2141  0.0125
21600.0000    -0.0476       PE 15.7931 15.8945 -0.1014
21700.0000    -0.0430       PE 15.5066 15.5809 -0.0743
21800.0000    -0.0384       PE 15.0415 15.2740 -0.2326
21900.0000    -0.0338       PE 14.7470 14.9751 -0.2281
22000.0000    -0.0293       PE 14.6685 14.6851 -0.0167
22100.0000    -0.0247       PE 14.2546 14.4055 -0.1509
22200.0000    -0.0202       PE 14.0986 14.1376 -0.0390
22300.0000    -0.0157       PE 13.8903 13.8829  0.0074
22400.0000    -0.0113       PE 13.6764 13.6430  0.0334
22500.0000    -0.0068       PE 13.4661 13.4197  0.0464
22600.00

## Layer 4 — trader-standard vol signals

Turns each SVI smile into the numbers you actually read: **ATM IV** (level), **skew** (ATM slope + `rr25`, the 25Δ risk reversal), **convexity** (`bf25`, 25Δ butterfly), and a **constant-maturity term structure** (7d/14d/30d ATM IV + slope).

Saves per-expiry signals to `vol_surface/signals/{symbol}_{date}.csv` and rebuilds the masters `_signals_all.csv` (per-expiry) and `_daily.csv` (one dashboard row per symbol/day).

How to read: `rr25 < 0` means puts are richer than calls (downside skew — normal); more negative = steeper fear. `bf25 > 0` means the wings are bid vs ATM (fat tails priced). `term_slope_raw > 0` = longer-dated vol above front (normal); `< 0` = inverted (near-term stress/event). Constant-maturity IVs are flagged `in_range=False` when no expiry brackets that tenor (e.g. BANKNIFTY has no weekly, so its 7d is extrapolated) — Layer 5 will lean on the in-range tenors and on per-expiry IV-vs-realised.

In [27]:
# ── Layer 4: distil SVI smiles into trader signals, persist ──────────
from src.vol_signals import expiry_signal_rows, daily_from_expiry_frame

SIG_DIR = os.path.join(VOL_DIR, "signals")
os.makedirs(SIG_DIR, exist_ok=True)

for sym in ACTIVE_SYMBOLS:
    svi_csv = os.path.join(SVI_DIR, f"{sym.lower()}_{TODAY}.csv")
    if not os.path.exists(svi_csv):
        print(f"{sym}: no SVI file for {TODAY}")
        continue
    rows, _ = expiry_signal_rows(pd.read_csv(svi_csv))
    if rows:
        pd.DataFrame(rows).to_csv(os.path.join(SIG_DIR, f"{sym.lower()}_{TODAY}.csv"), index=False)

# rebuild masters from all per-day CSVs so history stays consistent
per_day = [p for p in sorted(glob.glob(os.path.join(SIG_DIR, "*_*.csv")))
           if not os.path.basename(p).startswith("_")]
if per_day:
    expiry_master = pd.concat([pd.read_csv(p) for p in per_day], ignore_index=True)
    expiry_master = expiry_master.sort_values(["symbol", "date", "dte"])
    expiry_master.to_csv(os.path.join(SIG_DIR, "_signals_all.csv"), index=False)

    daily_master = daily_from_expiry_frame(expiry_master).sort_values(["symbol", "date"])
    daily_master.to_csv(os.path.join(SIG_DIR, "_daily.csv"), index=False)
    print(f"Saved vol signals -> {SIG_DIR}  ({len(expiry_master)} expiry rows, "
          f"{daily_master.shape[0]} symbol-days)")

Saved vol signals -> d:\nse-options-scraper\data\processed\vol_surface\signals  (312 expiry rows, 118 symbol-days)


In [28]:
# ── Layer 4 dashboard: today's signals ──
today_daily = daily_master[daily_master["date"] == TODAY]
show_daily = ["symbol", "front_dte", "front_atm_iv", "front_skew", "front_rr25",
              "front_bf25", "cmt7_iv", "cmt7_in_range", "cmt30_iv", "cmt30_in_range",
              "term_slope_raw"]
print("DAILY DASHBOARD (front expiry + constant-maturity)")
print(today_daily[show_daily].to_string(index=False))

print("\nPER-EXPIRY SIGNALS")
today_exp = expiry_master[expiry_master["date"] == TODAY]
show_exp = ["symbol", "expiry", "dte", "atm_iv", "atm_skew", "rr25", "bf25", "no_arb_ok"]
print(today_exp[show_exp].to_string(index=False))

DAILY DASHBOARD (front expiry + constant-maturity)
   symbol  front_dte  front_atm_iv  front_skew  front_rr25  front_bf25  cmt7_iv  cmt7_in_range  cmt30_iv  cmt30_in_range  term_slope_raw
BANKNIFTY         20       14.6542     -0.4529     -2.0835      0.3505  14.6542          False   14.1616            True         -0.8598
    NIFTY          6       11.8497     -0.5742     -1.1765      0.3039  12.0104           True   13.2752            True          1.3417

PER-EXPIRY SIGNALS
   symbol      expiry  dte  atm_iv  atm_skew    rr25   bf25  no_arb_ok
BANKNIFTY 27-Oct-2026   20 14.6542   -0.4529 -2.0835 0.3505      False
BANKNIFTY 23-Nov-2026   47 13.7944   -0.4210 -2.7833 0.4054       True
    NIFTY 13-Oct-2026    6 11.8497   -0.5742 -1.1765 0.3039       True
    NIFTY 19-Oct-2026   12 12.4032   -0.5429 -1.6680 0.3426       True
    NIFTY 27-Oct-2026   20 12.8942   -0.4732 -1.9440 0.3801       True
    NIFTY 03-Nov-2026   27 13.1131   -0.4166 -1.9852 0.3621       True
    NIFTY 09-Nov-2026

## Layer 5 — realized vol + variance risk premium (VRP)

Compares our implied IV against **realized volatility** from the index OHLC (`data/raw/underlying/`). Saves to `vol_surface/vrp/`:

- `per_expiry.csv` — implied ATM IV observed on day *t* vs realized vol over the path *t → expiry* (scores once the expiry passes). This is the rigorous, expiry-aligned VRP and doubles as the **IV-as-forecast** baseline.
- `_daily.csv` — fixed-horizon VRP (cmt7/cmt30 vs forward realized) plus a **contemporaneous gauge** (implied − trailing-20d realized), available every day.

**VRP = implied − realized** (vol points): positive = options were richer than what realized (premium-seller edge); negative = options were cheap (they under-priced the move). Realized is close-to-close (includes overnight gaps). With only a handful of completed expiry cycles — and overlapping observation windows — read this **directionally**, not as statistics yet.

In [29]:
# ── Layer 5: realized vol + VRP, persist ──────────
from src.vrp import build_per_expiry, build_daily, load_ohlc

UNDERLYING_DIR = os.path.join(PROJECT_ROOT, "data", "raw", "underlying")
VRP_DIR = os.path.join(VOL_DIR, "vrp")
os.makedirs(VRP_DIR, exist_ok=True)

ohlc = load_ohlc(UNDERLYING_DIR, ACTIVE_SYMBOLS)
vrp_daily = None
if not ohlc:
    print("No underlying OHLC found — run:  python scripts/fetch_underlying.py")
else:
    sig_all = pd.read_csv(os.path.join(SIG_DIR, "_signals_all.csv"))
    sig_daily = pd.read_csv(os.path.join(SIG_DIR, "_daily.csv"))

    per_exp = build_per_expiry(sig_all, ohlc).sort_values(["symbol", "obs_date", "dte"])
    per_exp.to_csv(os.path.join(VRP_DIR, "per_expiry.csv"), index=False)

    vrp_daily = build_daily(sig_daily, ohlc).sort_values(["symbol", "date"])
    vrp_daily.to_csv(os.path.join(VRP_DIR, "_daily.csv"), index=False)

    scored = per_exp[per_exp["completed"]]
    through = {s: ohlc[s]["date"].iloc[-1] for s in ohlc}
    print(f"OHLC through: {through}")
    print(f"VRP saved -> {VRP_DIR}  ({len(per_exp)} per-expiry rows, "
          f"{len(scored)} scored; {len(vrp_daily)} daily rows)")

OHLC through: {'NIFTY': '2026-10-07', 'BANKNIFTY': '2026-10-07'}
VRP saved -> d:\nse-options-scraper\data\processed\vol_surface\vrp  (312 per-expiry rows, 218 scored; 118 daily rows)


In [30]:
# ── Layer 5 dashboard ──
if vrp_daily is not None:
    today_g = vrp_daily[vrp_daily["date"] == TODAY]
    if not today_g.empty:
        print("TODAY — implied vs trailing-20d realized (rich/cheap gauge) + cmt7 VRP")
        print(today_g[["symbol", "front_atm_iv", "trailing_rv20_cc", "iv_minus_trail20",
                       "cmt7_iv", "rv_fwd7_cc", "vrp_cmt7", "cmt7_scored"]].to_string(index=False))

    scored = per_exp[per_exp["completed"]]
    if len(scored):
        print("\nVRP scored per-expiry (implied − realized), by symbol:")
        summ = scored.groupby("symbol").agg(
            n=("vrp_cc", "size"), mean_vrp_cc=("vrp_cc", "mean"),
            median_vrp_cc=("vrp_cc", "median"), seller_win_pct=("seller_win", "mean"))
        summ["seller_win_pct"] = (summ["seller_win_pct"] * 100).round(0)
        print(summ.round(2).to_string())

        print("\nMost recent 8 scored expiries (IV-as-forecast baseline):")
        cols = ["symbol", "obs_date", "expiry", "dte", "atm_iv", "rv_cc", "vrp_cc", "seller_win"]
        print(scored.sort_values("obs_date").tail(8)[cols].to_string(index=False))
    print("\nNote: overlapping windows + few completed cycles — directional read, not statistics yet.")

TODAY — implied vs trailing-20d realized (rich/cheap gauge) + cmt7 VRP
   symbol  front_atm_iv  trailing_rv20_cc  iv_minus_trail20  cmt7_iv  rv_fwd7_cc  vrp_cmt7  cmt7_scored
BANKNIFTY       14.6542           13.6222            1.0320  14.6542         NaN       NaN        False
    NIFTY       11.8497           12.1625           -0.3128  12.0104         NaN       NaN        False

VRP scored per-expiry (implied − realized), by symbol:
             n  mean_vrp_cc  median_vrp_cc seller_win_pct
symbol                                                   
BANKNIFTY   79       1.1400         1.5600        62.0253
NIFTY      139       1.2400         1.8900        69.0647

Most recent 8 scored expiries (IV-as-forecast baseline):
   symbol   obs_date      expiry  dte  atm_iv   rv_cc  vrp_cc seller_win
    NIFTY 2026-09-23 06-Oct-2026   13  8.5000 15.5034 -7.0034      False
    NIFTY 2026-09-24 06-Oct-2026   12 11.5013 13.2921 -1.7908      False
BANKNIFTY 2026-09-24 29-Sep-2026    5 14.3908 18.955